[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ronniewillaert/SPM-Textbook-Python/blob/main/notebooks/part-I-introduction/ch03/SPM_Ch3_AFM_Instrumentation_Python_Exercises.ipynb)

# AFM Instrumentation: From Detection to Measurement
## Interactive Companion to Chapter 3 — *Scanning Probe Microscopy*

This notebook accompanies **Chapter 3** of the textbook *Scanning Probe Microscopy*.

After completing this notebook you will be able to:

- Simulate cantilever force–deflection (Hooke's law) and understand force sensitivity.
- Explore the optical lever amplification and quadrant photodiode detection.
- Model piezoelectric displacement and hysteresis in AFM scanners.
- Simulate PI feedback control and understand feedback bandwidth.
- Quantify thermal noise as a fundamental sensitivity limit.
- Perform a complete deflection sensitivity calibration (voltage → deflection → force).
- Explore parameter trade-offs in cantilever selection for different applications.

The notebook contains **10 interactive exercises**, each with problem statements, runnable code, plots, and interactive sliders.

---


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ---------- Physical constants ----------
k_B   = 1.380649e-23    # Boltzmann constant        (J/K)
T_ref = 300              # Room temperature          (K)
e_0   = 1.602176634e-19 # Elementary charge         (C)
eps_0 = 8.8541878128e-12 # Vacuum permittivity      (F/m)

# ---------- Matplotlib defaults ----------
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 11

print("Physical constants loaded ✓")


Physical constants loaded ✓


In [2]:
# If widgets do not render, run this cell once (Colab).
try:
    from ipywidgets import interact, FloatLogSlider, FloatSlider, IntSlider, Dropdown, HTML, fixed
    import ipywidgets as widgets
    from IPython.display import display, Markdown, clear_output
    print("Widgets ready ✓")
except:
    print("ipywidgets not available, installing...")
    import subprocess
    subprocess.check_call(['pip', '-q', 'install', 'ipywidgets'])
    from ipywidgets import interact, FloatLogSlider, FloatSlider, IntSlider, Dropdown, HTML, fixed
    import ipywidgets as widgets
    from IPython.display import display, Markdown, clear_output
    print("Widgets ready ✓")


Widgets ready ✓


---
## 1. Cantilever Force–Deflection Simulation (Hooke's Law)

The AFM cantilever functions as a mechanical force sensor. When the probe interacts with a surface, the cantilever bends in response to the interaction force. To a good approximation, the cantilever behaves as a linear spring:

$$F = k \, \Delta z$$

where $F$ is the force acting on the cantilever, $k$ is the spring constant (N/m), and $\Delta z$ is the deflection (m).

**Parameters:**
- **k** (spring constant): ranges from ~0.01 N/m (soft, biological) to ~40 N/m (stiff, tapping mode)
- **Δz** (deflection): typically ±20 nm during imaging

**Key insight:** The slope of the force–deflection curve equals the spring constant. Softer cantilevers produce larger deflections for the same force, providing higher force sensitivity — but also higher susceptibility to noise.

### Tasks
1. Choose a cantilever spring constant $k$.
2. Generate deflections between −20 nm and +20 nm.
3. Compute the corresponding force.
4. Plot force vs. deflection and explore how stiffness changes the slope.


In [3]:
def interactive_hookes_law(k_N_per_m=0.1):
    z_nm = np.linspace(-20, 20, 500)
    z_m = z_nm * 1e-9
    F_N = k_N_per_m * z_m
    F_nN = F_N * 1e9

    print(f"  Cantilever Force–Deflection")
    print(f"  ─────────────────────────────────")
    print(f"  Spring constant  k = {k_N_per_m:.3g} N/m")
    print(f"  Deflection range   = ±20 nm")
    print()
    
    # Force at specific deflections
    for dz in [1, 5, 10, 20]:
        F_val = k_N_per_m * dz * 1e-9
        print(f"  At Δz = {dz:4d} nm:  F = {F_val:.3e} N  ({F_val*1e9:.3f} nN)")
    print()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    ax1.plot(z_nm, F_nN, 'b-', lw=2.5)
    ax1.axhline(0, color='gray', lw=0.5)
    ax1.axvline(0, color='gray', lw=0.5)
    ax1.set_xlabel('Deflection Δz (nm)')
    ax1.set_ylabel('Force F (nN)')
    ax1.set_title(f"Hooke's law: k = {k_N_per_m:.3g} N/m")

    # Compare three stiffnesses
    k_values = [0.01, 0.1, 40]
    for kv in k_values:
        F_comp = kv * z_m * 1e9
        ax2.plot(z_nm, F_comp, lw=2, label=f'k = {kv} N/m')
    ax2.axhline(0, color='gray', lw=0.5)
    ax2.set_xlabel('Deflection Δz (nm)')
    ax2.set_ylabel('Force F (nN)')
    ax2.set_title('Effect of cantilever stiffness')
    ax2.legend()

    plt.tight_layout()
    plt.show()

interact(
    interactive_hookes_law,
    k_N_per_m=FloatLogSlider(value=0.1, base=10, min=-2, max=2, step=0.1, description='k (N/m)')
);


interactive(children=(FloatLogSlider(value=0.1, description='k (N/m)', max=2.0, min=-2.0), Output()), _dom_cla…

---
## 2. Optical Lever Amplification

In optical beam deflection detection, a focused laser beam reflects from the back of the cantilever and is detected by a position-sensitive photodiode. When the cantilever tilts by a small angle $\theta$, the reflected beam changes direction by approximately $2\theta$.

The displacement of the laser spot on the photodiode is:

$$\Delta x \approx 2 L \theta$$

where $L$ is the distance between cantilever and photodiode, and $\theta$ is the cantilever tilt angle.

**Parameters:**
- **L** (detector distance): typically 2–10 cm in commercial AFMs
- **θ** (tilt angle): $10^{-7}$ to $10^{-5}$ rad during typical imaging

**Key insight:** The detector distance $L$ acts as a geometric amplification factor. Increasing $L$ amplifies the spot displacement without any electronic gain — this is the essence of the optical lever principle.

### Tasks
1. Define tilt angles between $10^{-7}$ and $10^{-5}$ rad.
2. Compute $\Delta x$ for detector distances of 2 cm, 5 cm, and 10 cm.
3. Plot $\Delta x$ vs $\theta$ and explore how $L$ amplifies the signal.


In [4]:
def interactive_optical_lever(L_cm=5.0):
    L_m = L_cm * 1e-2
    theta = np.logspace(-7, -5, 500)
    dx_m = 2 * L_m * theta
    dx_um = dx_m * 1e6

    print(f"  Optical Lever Amplification")
    print(f"  ─────────────────────────────────")
    print(f"  Detector distance  L = {L_cm:.1f} cm")
    print(f"  Amplification factor = 2L = {2*L_m*100:.1f} cm")
    print()
    
    for th in [1e-7, 1e-6, 1e-5]:
        dx = 2 * L_m * th
        print(f"  θ = {th:.0e} rad → Δx = {dx*1e6:.3f} µm  ({dx*1e9:.1f} nm)")
    print()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    ax1.loglog(theta, dx_um, 'b-', lw=2.5)
    ax1.set_xlabel('Cantilever tilt θ (rad)')
    ax1.set_ylabel('Spot displacement Δx (µm)')
    ax1.set_title(f'Optical lever: L = {L_cm:.1f} cm')

    # Compare distances
    Ls_cm = [2, 5, 10]
    for Lv in Ls_cm:
        dx_comp = 2 * (Lv * 1e-2) * theta * 1e6
        ax2.loglog(theta, dx_comp, lw=2, label=f'L = {Lv} cm')
    ax2.set_xlabel('Cantilever tilt θ (rad)')
    ax2.set_ylabel('Spot displacement Δx (µm)')
    ax2.set_title('Effect of detector distance')
    ax2.legend()

    plt.tight_layout()
    plt.show()

interact(
    interactive_optical_lever,
    L_cm=FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5, description='L (cm)')
);


interactive(children=(FloatSlider(value=5.0, description='L (cm)', max=20.0, min=1.0, step=0.5), Output()), _d…

---
## 3. Quadrant Photodiode Detection

The AFM uses a four-segment (quadrant) photodiode to detect the position of the reflected laser spot. The photodiode is divided into four quadrants: A (top-left), B (top-right), C (bottom-left), D (bottom-right).

Two differential signals are computed:

**Vertical deflection** (normal force):

$$V_{\text{vert}} = \frac{(I_A + I_B) - (I_C + I_D)}{I_A + I_B + I_C + I_D}$$

**Lateral deflection** (friction/torsion):

$$V_{\text{lat}} = \frac{(I_A + I_C) - (I_B + I_D)}{I_A + I_B + I_C + I_D}$$

**Key insight:** The normalized differential signals are insensitive to total laser intensity variations, providing robust detection of cantilever deflection and torsion simultaneously.

### Tasks
1. Simulate a Gaussian laser spot (~0.5 mm diameter) on a four-quadrant detector.
2. Move the spot vertically and laterally (offsets in mm).
3. Observe how the vertical and lateral signals change and where the linear detection range lies.



In [5]:
def interactive_photodiode(spot_y_mm=0.0, spot_x_mm=0.0, spot_diameter_mm=0.5):
    # Detector is 2 mm x 2 mm, divided into four quadrants
    detector_half = 1.0  # mm (half-size)
    spot_radius = spot_diameter_mm / 2  # mm

    # High-resolution 2D grid (in mm)
    x = np.linspace(-detector_half, detector_half, 400)
    y = np.linspace(-detector_half, detector_half, 400)
    X, Y = np.meshgrid(x, y)

    # Gaussian laser spot (all in mm)
    intensity = np.exp(-((X - spot_x_mm)**2 + (Y - spot_y_mm)**2) / (2 * spot_radius**2))

    # Quadrant intensities
    I_A = np.sum(intensity[(Y > 0) & (X < 0)])  # top-left
    I_B = np.sum(intensity[(Y > 0) & (X > 0)])  # top-right
    I_C = np.sum(intensity[(Y < 0) & (X < 0)])  # bottom-left
    I_D = np.sum(intensity[(Y < 0) & (X > 0)])  # bottom-right
    I_total = I_A + I_B + I_C + I_D

    if I_total > 0:
        V_vert = ((I_A + I_B) - (I_C + I_D)) / I_total
        V_lat  = ((I_A + I_C) - (I_B + I_D)) / I_total
    else:
        V_vert = V_lat = 0.0

    print(f"  Quadrant Photodiode Detection")
    print(f"  ─────────────────────────────────")
    print(f"  Spot position:   x = {spot_x_mm:+.2f} mm,  y = {spot_y_mm:+.2f} mm")
    print(f"  Spot diameter:   {spot_diameter_mm:.2f} mm")
    print()
    print(f"  Quadrant intensities (a.u.):")
    print(f"    A (top-L) = {I_A:.1f}    B (top-R) = {I_B:.1f}")
    print(f"    C (bot-L) = {I_C:.1f}    D (bot-R) = {I_D:.1f}")
    print()
    print(f"  Vertical signal  V_vert = {V_vert:+.4f}")
    print(f"  Lateral signal   V_lat  = {V_lat:+.4f}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    # Left: detector image with visible laser spot
    ax1.imshow(intensity, extent=[-detector_half, detector_half,
                                   -detector_half, detector_half],
               origin='lower', cmap='hot', aspect='equal',
               vmin=0, vmax=1)
    ax1.axhline(0, color='white', lw=1, ls='--')
    ax1.axvline(0, color='white', lw=1, ls='--')
    ax1.set_xlabel('x (mm)')
    ax1.set_ylabel('y (mm)')
    ax1.set_title('Laser spot on photodiode')
    ax1.text(-0.5, 0.7, 'A', color='white', fontsize=14, ha='center', fontweight='bold')
    ax1.text( 0.5, 0.7, 'B', color='white', fontsize=14, ha='center', fontweight='bold')
    ax1.text(-0.5,-0.7, 'C', color='white', fontsize=14, ha='center', fontweight='bold')
    ax1.text( 0.5,-0.7, 'D', color='white', fontsize=14, ha='center', fontweight='bold')

    # Right: vertical signal vs. vertical spot displacement
    offsets_mm = np.linspace(-1.0, 1.0, 300)
    V_vert_sweep = []
    for off in offsets_mm:
        I_s = np.exp(-((X - spot_x_mm)**2 + (Y - off)**2) / (2 * spot_radius**2))
        IA = np.sum(I_s[(Y > 0) & (X < 0)])
        IB = np.sum(I_s[(Y > 0) & (X > 0)])
        IC = np.sum(I_s[(Y < 0) & (X < 0)])
        ID = np.sum(I_s[(Y < 0) & (X > 0)])
        It = IA + IB + IC + ID
        V_vert_sweep.append(((IA+IB)-(IC+ID))/It if It > 0 else 0)

    ax2.plot(offsets_mm, V_vert_sweep, 'b-', lw=2)
    ax2.axhline(0, color='gray', lw=0.5)
    ax2.axvline(spot_y_mm, color='red', ls='--', lw=1.5,
                label=f'Current: {spot_y_mm:+.2f} mm')
    ax2.set_xlabel('Vertical spot offset (mm)')
    ax2.set_ylabel('Vertical signal V_vert')
    ax2.set_title('Photodiode response curve')
    ax2.legend()

    plt.tight_layout()
    plt.show()

interact(
    interactive_photodiode,
    spot_y_mm=FloatSlider(value=0.0, min=-0.8, max=0.8, step=0.02,
                          description='y offset (mm)', readout_format='.2f'),
    spot_x_mm=FloatSlider(value=0.0, min=-0.8, max=0.8, step=0.02,
                          description='x offset (mm)', readout_format='.2f'),
    spot_diameter_mm=FloatSlider(value=0.5, min=0.1, max=1.5, step=0.05,
                                description='Spot ⌀ (mm)', readout_format='.2f')
);


interactive(children=(FloatSlider(value=0.0, description='y offset (µm)', max=300.0, min=-300.0, step=5.0), Fl…

---
## 4. Piezoelectric Scanner — Voltage to Displacement

AFM scanners use piezoelectric actuators that convert applied voltage into nanometer-scale displacement:

$$\Delta x = d \cdot V$$

where $d$ is the piezoelectric coefficient (m/V) and $V$ is the applied voltage.

For typical PZT ceramics used in AFM:
- **d** ≈ 0.1 nm/V (for a single layer)
- Stack actuators multiply displacement by the number of layers $N$: $\Delta x = N \cdot d \cdot V$
- Tube scanners bend laterally with: $\Delta x_{\text{lat}} \propto V \cdot L^2 / (d_{\text{tube}} \cdot t)$

**Key insight:** Piezoelectric actuators provide sub-angstrom positioning precision, enabling the nanometer-scale scanning required for AFM imaging.

### Tasks
1. Calculate displacement vs. voltage for different piezo coefficients.
2. Compare single-layer and stack actuators.
3. Explore the scan range achievable with typical AFM voltages.


In [6]:
def interactive_piezo(d_nm_per_V=0.1, N_layers=1, V_max=200):
    V = np.linspace(0, V_max, 500)
    dx_single = d_nm_per_V * V          # single layer (nm)
    dx_stack = N_layers * d_nm_per_V * V  # stack actuator (nm)

    print(f"  Piezoelectric Displacement")
    print(f"  ─────────────────────────────────")
    print(f"  Piezo coefficient    d = {d_nm_per_V:.3f} nm/V")
    print(f"  Number of layers     N = {N_layers}")
    print(f"  Maximum voltage      V = {V_max:.0f} V")
    print(f"  Max displacement (1 layer)  = {d_nm_per_V * V_max:.1f} nm")
    print(f"  Max displacement ({N_layers} layers) = {N_layers * d_nm_per_V * V_max:.1f} nm ({N_layers * d_nm_per_V * V_max / 1000:.2f} µm)")
    print()
    
    for Vt in [1, 10, 50, 100]:
        if Vt <= V_max:
            dx = N_layers * d_nm_per_V * Vt
            print(f"  V = {Vt:4d} V → Δx = {dx:8.2f} nm  ({dx/1000:.3f} µm)")
    print()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    ax1.plot(V, dx_single, 'b-', lw=2, label='1 layer')
    if N_layers > 1:
        ax1.plot(V, dx_stack, 'r-', lw=2, label=f'{N_layers} layers')
    ax1.set_xlabel('Voltage (V)')
    ax1.set_ylabel('Displacement (nm)')
    ax1.set_title('Piezoelectric displacement: Δx = N·d·V')
    ax1.legend()

    # Compare different d values
    d_values = [0.05, 0.1, 0.2, 0.5]
    for dv in d_values:
        dx_comp = N_layers * dv * V
        ax2.plot(V, dx_comp / 1000, lw=2, label=f'd = {dv} nm/V')
    ax2.set_xlabel('Voltage (V)')
    ax2.set_ylabel('Displacement (µm)')
    ax2.set_title(f'Effect of piezo coefficient (N = {N_layers})')
    ax2.legend()

    plt.tight_layout()
    plt.show()

interact(
    interactive_piezo,
    d_nm_per_V=FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01, description='d (nm/V)', readout_format='.3f'),
    N_layers=IntSlider(value=1, min=1, max=200, step=1, description='N layers'),
    V_max=FloatSlider(value=200, min=10, max=400, step=10, description='V_max (V)')
);


interactive(children=(FloatSlider(value=0.1, description='d (nm/V)', max=1.0, min=0.01, readout_format='.3f', …

---
## 5. Piezoelectric Hysteresis and Creep

Piezoelectric actuators exhibit **hysteresis**: the displacement for a given voltage depends on the previous voltage history. The forward (increasing V) and return (decreasing V) paths do not coincide.

A simple phenomenological model uses different effective piezo coefficients for the forward and return paths:

$$\Delta x_{\text{forward}} = d \cdot V$$
$$\Delta x_{\text{return}} = d \cdot V + H \cdot d \cdot V_{\max} \cdot \sin\!\left(\pi \cdot \frac{V}{V_{\max}}\right)$$

where $H$ is the hysteresis parameter (typically 5–20% for open-loop PZT scanners).

**Creep** is a time-dependent drift: after a voltage step, the actuator slowly continues to deform.

**Key insight:** Hysteresis introduces positioning errors that distort AFM images. Closed-loop scanners with position sensors compensate for this effect.

### Tasks
1. Simulate a voltage ramp (up and down) and observe hysteresis.
2. Vary the hysteresis parameter to see its effect on positioning accuracy.
3. Understand why closed-loop correction is needed for accurate imaging.


In [7]:
def interactive_hysteresis(d_nm_per_V=0.1, V_max=150, hysteresis_pct=10):
    H = hysteresis_pct / 100.0
    
    # Forward sweep (0 → V_max)
    V_fwd = np.linspace(0, V_max, 500)
    dx_fwd = d_nm_per_V * V_fwd
    
    # Return sweep (V_max → 0)
    V_ret = np.linspace(V_max, 0, 500)
    dx_ret = d_nm_per_V * V_ret + H * d_nm_per_V * V_max * np.sin(np.pi * V_ret / V_max)
    
    # Maximum hysteresis error
    V_mid = V_max / 2
    dx_fwd_mid = d_nm_per_V * V_mid
    dx_ret_mid = d_nm_per_V * V_mid + H * d_nm_per_V * V_max * np.sin(np.pi * 0.5)
    error_nm = dx_ret_mid - dx_fwd_mid
    
    print(f"  Piezoelectric Hysteresis")
    print(f"  ─────────────────────────────────")
    print(f"  Piezo coefficient    d = {d_nm_per_V:.3f} nm/V")
    print(f"  Maximum voltage      V = {V_max:.0f} V")
    print(f"  Hysteresis parameter H = {hysteresis_pct:.0f}%")
    print(f"  Total scan range       = {d_nm_per_V * V_max:.1f} nm")
    print(f"  Max hysteresis error   = {error_nm:.2f} nm ({error_nm/(d_nm_per_V*V_max)*100:.1f}% of range)")
    print()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    ax1.plot(V_fwd, dx_fwd, 'b-', lw=2.5, label='Forward (extend)')
    ax1.plot(V_ret, dx_ret, 'r-', lw=2.5, label='Return (retract)')
    ax1.fill_between(V_fwd, dx_fwd, np.interp(V_fwd, V_ret[::-1], dx_ret[::-1]), alpha=0.15, color='red')
    ax1.set_xlabel('Voltage (V)')
    ax1.set_ylabel('Displacement (nm)')
    ax1.set_title(f'Piezo hysteresis loop (H = {hysteresis_pct}%)')
    ax1.legend()

    # Compare different hysteresis values
    H_values = [2, 5, 10, 20]
    for Hv in H_values:
        Hf = Hv / 100.0
        V_r = np.linspace(V_max, 0, 500)
        dx_r = d_nm_per_V * V_r + Hf * d_nm_per_V * V_max * np.sin(np.pi * V_r / V_max)
        ax2.plot(V_r, dx_r - np.interp(V_r, V_fwd, dx_fwd), lw=2, label=f'H = {Hv}%')
    ax2.set_xlabel('Voltage (V)')
    ax2.set_ylabel('Positioning error (nm)')
    ax2.set_title('Hysteresis error vs. H parameter')
    ax2.legend()

    plt.tight_layout()
    plt.show()

interact(
    interactive_hysteresis,
    d_nm_per_V=FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01, description='d (nm/V)', readout_format='.3f'),
    V_max=FloatSlider(value=150, min=10, max=400, step=10, description='V_max (V)'),
    hysteresis_pct=FloatSlider(value=10, min=0, max=25, step=0.5, description='H (%)')
);


interactive(children=(FloatSlider(value=0.1, description='d (nm/V)', max=0.5, min=0.01, readout_format='.3f', …

---
## 6. PI Feedback Control Simulation

During AFM imaging, a **Proportional–Integral (PI) controller** continuously adjusts the z-piezo position to maintain a constant tip–sample interaction.

The error signal is:

$$e(t) = s_{\text{setpoint}} - s_{\text{measured}}(t)$$

The PI controller output (driving the z-piezo) is:

$$u(t) = K_P \, e(t) + K_I \int_0^t e(\tau) \, d\tau$$

- **P-term** ($K_P$): provides immediate, proportional correction — fast response to topographic changes.
- **I-term** ($K_I$): accumulates error over time — eliminates steady-state offset and compensates drift.

**Key insight:** Too-low gains cause blurred images (slow tracking). Too-high gains cause oscillations and noise. Optimal imaging requires balancing responsiveness and stability.

### Tasks
1. Simulate the feedback response to a step-like surface feature.
2. Vary $K_P$ and $K_I$ to observe underdamped, overdamped, and optimal behavior.
3. Understand why PI control is sufficient for most AFM applications.


In [8]:
def interactive_pi_feedback(K_P=1.0, K_I=5.0, step_height_nm=10.0, scan_speed=1.0):
    '''
    Simulate PI feedback tracking a step feature on the surface.
    '''
    dt = 0.0005
    N = 4000
    t = np.arange(N) * dt

    # Surface profile: step feature
    surface = np.zeros(N)
    step_start = int(0.25 * N)
    step_end = int(0.75 * N)
    surface[step_start:step_end] = step_height_nm

    # PI controller simulation
    z_piezo = np.zeros(N)
    error = np.zeros(N)
    deflection_arr = np.zeros(N)
    integral = 0.0

    # Response rate scales with scan speed
    response_rate = 0.3 * scan_speed

    for i in range(1, N):
        # Cantilever deflection = what the feedback hasn't yet corrected
        deflection = surface[i] - z_piezo[i-1]
        deflection_arr[i] = deflection

        # Error signal (want deflection = 0)
        error[i] = deflection

        # PI controller accumulates and corrects
        integral += error[i] * dt
        correction = K_P * error[i] + K_I * integral

        # Z-piezo responds toward the correction target
        z_piezo[i] = z_piezo[i-1] + response_rate * (correction - z_piezo[i-1])

    tracking_error = surface - z_piezo
    rms_error = np.sqrt(np.mean(tracking_error[step_start:step_end]**2))

    print(f"  PI Feedback Control Simulation")
    print(f"  ─────────────────────────────────")
    print(f"  Proportional gain  K_P = {K_P:.2f}")
    print(f"  Integral gain      K_I = {K_I:.2f}")
    print(f"  Step height             = {step_height_nm:.1f} nm")
    print(f"  RMS tracking error      = {rms_error:.3f} nm")
    print()

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0,0].plot(t, surface, 'k--', lw=1.5, label='Surface (true)')
    axes[0,0].plot(t, z_piezo, 'b-', lw=2, label='Z-piezo (tracked)')
    axes[0,0].set_ylabel('Height (nm)')
    axes[0,0].set_title('Topography tracking')
    axes[0,0].legend()

    axes[0,1].plot(t, tracking_error, 'r-', lw=1.5)
    axes[0,1].axhline(0, color='gray', lw=0.5)
    axes[0,1].set_ylabel('Tracking error (nm)')
    axes[0,1].set_title('Error signal (surface − z_piezo)')

    axes[1,0].plot(t, deflection_arr, 'g-', lw=1.5)
    axes[1,0].axhline(0, color='gray', lw=0.5)
    axes[1,0].set_xlabel('Time (a.u.)')
    axes[1,0].set_ylabel('Cantilever deflection (nm)')
    axes[1,0].set_title('Residual deflection')

    # Compare different gain settings
    gains = [(0.1, 0.5, 'Low gains (slow)'), (1.0, 5.0, 'Medium gains'), (5.0, 30.0, 'High gains (fast)')]
    for kp, ki, lab in gains:
        zp = np.zeros(N)
        integ = 0.0
        for i in range(1, N):
            defl = surface[i] - zp[i-1]
            integ += defl * dt
            corr = kp * defl + ki * integ
            zp[i] = zp[i-1] + response_rate * (corr - zp[i-1])
        axes[1,1].plot(t, zp, lw=2, label=lab)
    axes[1,1].plot(t, surface, 'k--', lw=1, label='Surface')
    axes[1,1].set_xlabel('Time (a.u.)')
    axes[1,1].set_ylabel('Height (nm)')
    axes[1,1].set_title('Effect of gain settings')
    axes[1,1].legend(fontsize=9)

    plt.tight_layout()
    plt.show()

interact(
    interactive_pi_feedback,
    K_P=FloatSlider(value=1.0, min=0.01, max=10, step=0.1, description='K_P'),
    K_I=FloatSlider(value=5.0, min=0.1, max=50, step=0.5, description='K_I'),
    step_height_nm=FloatSlider(value=10.0, min=1, max=50, step=1, description='Step (nm)'),
    scan_speed=FloatSlider(value=1.0, min=0.1, max=5, step=0.1, description='Speed')
);


interactive(children=(FloatSlider(value=1.0, description='K_P', max=10.0, min=0.01), FloatSlider(value=5.0, de…

---
## 7. Feedback Bandwidth as a Low-Pass Filter

The AFM feedback system behaves as a **low-pass filter**: only surface features with spatial frequencies below the feedback bandwidth can be faithfully tracked.

If the scan speed is $v$ and a surface feature has wavelength $\lambda$, the effective frequency seen by the feedback is:

$$f_{\text{feature}} = \frac{v}{\lambda}$$

The feedback can track features only when $f_{\text{feature}} < f_{\text{bandwidth}}$.

The critical scan speed is:

$$v_{\text{max}} = f_{\text{bandwidth}} \times \lambda_{\text{min}}$$

**Key insight:** Increasing scan speed without increasing bandwidth causes image distortion — fine features are smoothed out while coarse features remain accurate.

### Tasks
1. Simulate feedback response to surfaces with different spatial frequencies.
2. Vary scan speed and observe when features are lost.
3. Determine the maximum scan speed for a given feature size and bandwidth.


In [9]:
from scipy.signal import butter, filtfilt

def interactive_bandwidth(f_bw_kHz=1.0, scan_speed_um_s=10, feature_nm=50):
    '''
    Simulate feedback bandwidth as low-pass filtering of surface topography.
    '''
    f_bw = f_bw_kHz * 1e3  # Hz
    v_scan = scan_speed_um_s * 1e-6  # m/s
    lam_feature = feature_nm * 1e-9  # m
    
    # Effective frequency of the feature
    f_feature = v_scan / lam_feature
    
    # Spatial range: 2 µm scan
    scan_length_um = 2.0
    x_um = np.linspace(0, scan_length_um, 2000)
    x_m = x_um * 1e-6
    
    # Surface: combination of coarse and fine features
    coarse_nm = 5 * np.sin(2 * np.pi * x_m / (500e-9))       # 500 nm period
    fine_nm = 3 * np.sin(2 * np.pi * x_m / (feature_nm * 1e-9))  # user-defined period
    surface_nm = coarse_nm + fine_nm
    
    # Low-pass filter the surface (simulate finite bandwidth)
    from scipy.signal import butter, filtfilt
    # Convert spatial to temporal: t = x / v_scan
    dx = x_m[1] - x_m[0]
    dt = dx / v_scan if v_scan > 0 else 1e-6
    fs = 1 / dt  # sampling frequency
    
    f_cutoff = f_bw
    if f_cutoff < fs / 2:
        b, a = butter(2, f_cutoff / (fs / 2), btype='low')
        tracked_nm = filtfilt(b, a, surface_nm)
    else:
        tracked_nm = surface_nm.copy()
    
    ratio = f_feature / f_bw if f_bw > 0 else float('inf')
    trackable = "YES ✓" if ratio < 1 else "NO ✗ (feature lost)"
    
    print(f"  Feedback Bandwidth Analysis")
    print(f"  ─────────────────────────────────")
    print(f"  Feedback bandwidth     = {f_bw_kHz:.1f} kHz")
    print(f"  Scan speed             = {scan_speed_um_s:.1f} µm/s")
    print(f"  Feature wavelength     = {feature_nm:.0f} nm")
    print(f"  Feature frequency      = {f_feature:.0f} Hz  ({f_feature/1000:.2f} kHz)")
    print(f"  f_feature / f_bandwidth = {ratio:.2f}")
    print(f"  Feature trackable?     → {trackable}")
    print()
    
    v_max = f_bw * lam_feature * 1e6  # µm/s
    print(f"  Max scan speed for this feature = {v_max:.2f} µm/s")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    ax1.plot(x_um, surface_nm, 'k-', lw=1, label='True surface', alpha=0.7)
    ax1.plot(x_um, tracked_nm, 'b-', lw=2.5, label='Feedback-tracked')
    ax1.set_xlabel('Position (µm)')
    ax1.set_ylabel('Height (nm)')
    ax1.set_title(f'Topography tracking (v = {scan_speed_um_s} µm/s)')
    ax1.legend()

    # Right: frequency response
    speeds = [1, 5, 10, 50, 100]
    feature_sizes = np.logspace(1, 3, 200)  # nm
    ax2.axhline(f_bw_kHz, color='red', ls='--', lw=2, label=f'Bandwidth = {f_bw_kHz} kHz')
    for vs in speeds:
        f_feat = (vs * 1e-6) / (feature_sizes * 1e-9) / 1000  # kHz
        ax2.loglog(feature_sizes, f_feat, lw=1.5, label=f'v = {vs} µm/s')
    ax2.set_xlabel('Feature wavelength (nm)')
    ax2.set_ylabel('Effective frequency (kHz)')
    ax2.set_title('Feature frequency vs. bandwidth')
    ax2.legend(fontsize=8)
    ax2.set_ylim(0.01, 1000)

    plt.tight_layout()
    plt.show()

interact(
    interactive_bandwidth,
    f_bw_kHz=FloatSlider(value=1.0, min=0.1, max=10, step=0.1, description='BW (kHz)'),
    scan_speed_um_s=FloatSlider(value=10, min=0.5, max=100, step=0.5, description='v (µm/s)'),
    feature_nm=FloatSlider(value=50, min=10, max=500, step=5, description='λ (nm)')
);



interactive(children=(FloatSlider(value=1.0, description='BW (kHz)', max=10.0, min=0.1), FloatSlider(value=10.…

---
## 8. Thermal Noise of an AFM Cantilever

Even in the absence of external forces, AFM cantilevers exhibit random motion caused by thermal fluctuations. The **equipartition theorem** predicts:

$$\frac{1}{2} k \langle z^2 \rangle = \frac{1}{2} k_B T$$

leading to the mean-square thermal deflection:

$$\langle z^2 \rangle = \frac{k_B T}{k}$$

and the root-mean-square (RMS) thermal amplitude:

$$z_{\text{rms}} = \sqrt{\frac{k_B T}{k}}$$

**Parameters:**
- **k** (spring constant): 0.01–40 N/m
- **T** (temperature): typically 300 K (room temperature)

**Key insight:** Thermal noise sets a **fundamental limit** on AFM force sensitivity. Softer cantilevers have larger thermal fluctuations — increasing both signal sensitivity and noise. The minimum detectable force is $F_{\text{min}} \approx k \cdot z_{\text{rms}}$.

### Tasks
1. Compute $z_{\text{rms}}$ for different spring constants.
2. Simulate thermal fluctuations as Gaussian noise.
3. Compare soft vs. stiff cantilevers and understand the sensitivity trade-off.


In [10]:
def interactive_thermal_noise(k_N_per_m=0.2, T_K=300.0, n_samples=30000):
    z2 = k_B * T_K / k_N_per_m
    z_rms_m = np.sqrt(z2)
    z_rms_nm = z_rms_m * 1e9
    F_min = k_N_per_m * z_rms_m  # minimum detectable force

    # Simulated displacements
    z = np.random.normal(loc=0.0, scale=z_rms_m, size=int(n_samples))

    print(f"  Thermal Noise of AFM Cantilever")
    print(f"  ─────────────────────────────────")
    print(f"  Spring constant  k = {k_N_per_m:.4g} N/m")
    print(f"  Temperature      T = {T_K:.1f} K")
    print(f"  ⟨z²⟩              = {z2:.3e} m²")
    print(f"  z_rms              = {z_rms_nm:.4g} nm")
    print(f"  F_min = k·z_rms    = {F_min:.3e} N  ({F_min*1e12:.2f} pN)")
    print()
    
    # Compare several cantilevers
    k_examples = [0.01, 0.1, 1.0, 10, 40]
    print(f"  {'k (N/m)':>10s}   {'z_rms (nm)':>12s}   {'F_min (pN)':>12s}")
    print(f"  {'─'*10}   {'─'*12}   {'─'*12}")
    for kv in k_examples:
        zr = np.sqrt(k_B * T_K / kv) * 1e9
        fm = kv * np.sqrt(k_B * T_K / kv) * 1e12
        print(f"  {kv:10.3g}   {zr:12.4f}   {fm:12.3f}")
    print()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # Histogram
    axes[0].hist(z * 1e9, bins=80, density=True, color='steelblue', alpha=0.7, edgecolor='white')
    # Theoretical Gaussian overlay
    z_plot = np.linspace(-4*z_rms_nm, 4*z_rms_nm, 300)
    gauss = (1/(z_rms_nm * np.sqrt(2*np.pi))) * np.exp(-z_plot**2 / (2*z_rms_nm**2))
    axes[0].plot(z_plot, gauss, 'r-', lw=2, label='Theory')
    axes[0].axvline(-z_rms_nm, color='orange', ls='--', lw=1.5, label=f'±z_rms = ±{z_rms_nm:.3f} nm')
    axes[0].axvline(z_rms_nm, color='orange', ls='--', lw=1.5)
    axes[0].set_xlabel('Thermal deflection z (nm)')
    axes[0].set_ylabel('Probability density')
    axes[0].set_title('Thermal noise distribution')
    axes[0].legend(fontsize=9)

    # z_rms vs k
    k_range = np.logspace(-2, 2, 200)
    z_rms_range = np.sqrt(k_B * T_K / k_range) * 1e9
    axes[1].loglog(k_range, z_rms_range, 'b-', lw=2.5)
    axes[1].axhline(z_rms_nm, color='red', ls='--', lw=1)
    axes[1].axvline(k_N_per_m, color='red', ls='--', lw=1)
    axes[1].set_xlabel('Spring constant k (N/m)')
    axes[1].set_ylabel('z_rms (nm)')
    axes[1].set_title('Thermal amplitude vs. stiffness')

    # F_min vs k
    F_min_range = k_range * np.sqrt(k_B * T_K / k_range) * 1e12  # pN
    axes[2].loglog(k_range, F_min_range, 'g-', lw=2.5)
    axes[2].axvline(k_N_per_m, color='red', ls='--', lw=1)
    axes[2].set_xlabel('Spring constant k (N/m)')
    axes[2].set_ylabel('F_min (pN)')
    axes[2].set_title('Minimum detectable force')

    plt.tight_layout()
    plt.show()

interact(
    interactive_thermal_noise,
    k_N_per_m=FloatLogSlider(value=0.2, base=10, min=-2, max=2, step=0.1, description='k (N/m)'),
    T_K=FloatSlider(value=300.0, min=4.0, max=400.0, step=1.0, description='T (K)'),
    n_samples=IntSlider(value=30000, min=5000, max=100000, step=5000, description='Samples')
);


interactive(children=(FloatLogSlider(value=0.2, description='k (N/m)', max=2.0, min=-2.0), FloatSlider(value=3…

---
## 9. Deflection Sensitivity Calibration — From Voltage to Force

The AFM does not measure force directly. It measures a **voltage** on the photodiode, which must be converted through a calibration chain:

$$\text{Voltage } V_{\text{PD}} \;\xrightarrow{\; S \;}\; \text{Deflection } \Delta z \;\xrightarrow{\; k \;}\; \text{Force } F$$

**Step 1 — Deflection sensitivity** ($S$, in nm/V): determined by pressing the cantilever onto a hard surface.

$$\Delta z = S \times V_{\text{PD}}$$

**Step 2 — Force** ($F$): using Hooke's law with the calibrated spring constant.

$$F = k \times \Delta z = k \times S \times V_{\text{PD}}$$

**Key insight:** Errors in either $S$ or $k$ propagate directly into force measurements. A 10% error in deflection sensitivity causes a 10% error in all measured forces.

### Tasks
1. Simulate a force–distance curve on a hard surface to extract deflection sensitivity.
2. Apply the calibration chain to convert raw photodiode voltage into force.
3. Explore how calibration errors affect force measurements.


In [11]:
def interactive_calibration(S_nm_per_V=30.0, k_N_per_m=0.5, S_error_pct=0, k_error_pct=0):
    '''
    Simulate deflection sensitivity calibration and the V → Δz → F chain.
    '''
    # Simulate a force-distance curve on a hard surface
    # Z-piezo position (nm) — approach from far to contact
    z_piezo = np.linspace(100, -50, 1000)  # nm
    
    # On hard surface: once in contact, cantilever deflection = z_piezo displacement
    # Non-contact region: deflection ≈ 0
    contact_point = 0  # nm
    deflection_nm = np.where(z_piezo < contact_point, -z_piezo + contact_point, 0)
    
    # True photodiode voltage (using true S)
    V_pd = deflection_nm / S_nm_per_V  # V
    
    # Add small noise
    V_pd_noisy = V_pd + np.random.normal(0, 0.005, len(V_pd))
    
    # Calibration: fit the contact region to get S
    contact_mask = z_piezo < -5  # well into contact
    slope = np.polyfit(z_piezo[contact_mask], V_pd_noisy[contact_mask], 1)
    S_measured = -1.0 / slope[0]  # nm/V (inverse of V/nm slope)
    
    # Apply calibration with potential errors
    S_used = S_nm_per_V * (1 + S_error_pct / 100)
    k_used = k_N_per_m * (1 + k_error_pct / 100)
    
    # Example measurement: convert a voltage reading to force
    V_example = 0.2  # V
    dz_true = S_nm_per_V * V_example  # nm
    F_true = k_N_per_m * dz_true * 1e-9  # N
    
    dz_meas = S_used * V_example  # nm
    F_meas = k_used * dz_meas * 1e-9  # N
    
    F_error_pct = (F_meas - F_true) / F_true * 100 if F_true != 0 else 0
    
    print(f"  Deflection Sensitivity Calibration")
    print(f"  ─────────────────────────────────")
    print(f"  True S = {S_nm_per_V:.1f} nm/V    Used S = {S_used:.1f} nm/V  (error: {S_error_pct:+.0f}%)")
    print(f"  True k = {k_N_per_m:.3f} N/m   Used k = {k_used:.3f} N/m  (error: {k_error_pct:+.0f}%)")
    print(f"  Fitted S from curve = {S_measured:.2f} nm/V")
    print()
    print(f"  Example: V_PD = {V_example} V")
    print(f"  ─────────────────────────────────")
    print(f"  True:     Δz = {dz_true:.1f} nm,  F = {F_true*1e9:.3f} nN")
    print(f"  Measured:  Δz = {dz_meas:.1f} nm,  F = {F_meas*1e9:.3f} nN")
    print(f"  Force error = {F_error_pct:+.1f}%")
    print()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # Force-distance curve (voltage vs z-piezo)
    axes[0].plot(z_piezo, V_pd_noisy, 'b.', ms=0.5, alpha=0.5)
    axes[0].plot(z_piezo, V_pd, 'r-', lw=2, label='Ideal')
    axes[0].axvline(contact_point, color='green', ls='--', lw=1.5, label='Contact point')
    axes[0].set_xlabel('Z-piezo position (nm)')
    axes[0].set_ylabel('Photodiode voltage (V)')
    axes[0].set_title('Force curve on hard surface')
    axes[0].legend()
    axes[0].invert_xaxis()

    # Calibrated force-distance curve
    F_calibrated = k_used * (S_used * V_pd_noisy * 1e-9)  # N → nN
    F_true_curve = k_N_per_m * (S_nm_per_V * V_pd * 1e-9)
    axes[1].plot(z_piezo, F_true_curve * 1e9, 'k--', lw=1.5, label='True force')
    axes[1].plot(z_piezo, F_calibrated * 1e9, 'b-', lw=2, label='Calibrated force')
    axes[1].set_xlabel('Z-piezo position (nm)')
    axes[1].set_ylabel('Force (nN)')
    axes[1].set_title('Calibrated force curve')
    axes[1].legend()
    axes[1].invert_xaxis()

    # Error propagation
    S_errors = np.linspace(-20, 20, 100)
    k_errors = np.linspace(-20, 20, 100)
    S_err_grid, K_err_grid = np.meshgrid(S_errors, k_errors)
    F_err_grid = (1 + S_err_grid/100) * (1 + K_err_grid/100) * 100 - 100  # total % error
    
    cs = axes[2].contourf(S_errors, k_errors, F_err_grid, levels=20, cmap='RdBu_r')
    axes[2].plot(S_error_pct, k_error_pct, 'ko', ms=10)
    axes[2].set_xlabel('S error (%)')
    axes[2].set_ylabel('k error (%)')
    axes[2].set_title('Force error (%)')
    plt.colorbar(cs, ax=axes[2], label='Force error (%)')

    plt.tight_layout()
    plt.show()

interact(
    interactive_calibration,
    S_nm_per_V=FloatSlider(value=30, min=5, max=100, step=1, description='S (nm/V)'),
    k_N_per_m=FloatLogSlider(value=0.5, base=10, min=-2, max=2, step=0.1, description='k (N/m)'),
    S_error_pct=FloatSlider(value=0, min=-20, max=20, step=1, description='S error (%)'),
    k_error_pct=FloatSlider(value=0, min=-20, max=20, step=1, description='k error (%)')
);


interactive(children=(FloatSlider(value=30.0, description='S (nm/V)', min=5.0, step=1.0), FloatLogSlider(value…

---
## 10. Parameter Exploration — Cantilever Selection for Different Applications

Selecting the right cantilever is one of the most important experimental decisions in AFM. The choice involves trade-offs between:

- **Force sensitivity** (favoring soft cantilevers, low $k$)
- **Thermal noise** (favoring stiff cantilevers, high $k$)
- **Signal-to-noise ratio** (optimal at intermediate $k$)
- **Resonance frequency** (higher for smaller, stiffer cantilevers)
- **Application requirements** (biological samples need soft; hard materials need stiff)

The resonance frequency of a rectangular cantilever is approximately:

$$f_0 = \frac{1}{2\pi} \sqrt{\frac{k}{m^*}}$$

where $m^* \approx 0.24 \, \rho \, w \, t \, L$ is the effective mass.

### Tasks
1. Compare cantilever properties across a range of spring constants.
2. Identify the optimal cantilever for different applications.
3. Explore the sensitivity vs. noise trade-off quantitatively.


In [12]:
def interactive_cantilever_selection(application='Biological (contact)'):
    '''
    Compare cantilever properties and recommend optimal choice.
    '''
    # Cantilever database (typical commercial probes)
    cantilevers = {
        'MLCT-C (soft)':      {'k': 0.01,  'f0': 7,    'L': 320, 'use': 'Bio contact'},
        'SNL-A':              {'k': 0.35,  'f0': 65,   'L': 120, 'use': 'Bio contact'},
        'ScanAsyst-Air':      {'k': 0.4,   'f0': 70,   'L': 115, 'use': 'PeakForce'},
        'RFESPA-75':          {'k': 3.0,   'f0': 75,   'L': 225, 'use': 'Tapping'},
        'TESP':               {'k': 42,    'f0': 320,  'L': 125, 'use': 'Tapping hard'},
        'PPP-NCH':            {'k': 42,    'f0': 330,  'L': 125, 'use': 'NC-AFM'},
    }
    
    # Application requirements
    apps = {
        'Biological (contact)':     {'k_range': (0.005, 0.5),  'desc': 'Soft samples, minimize damage'},
        'Polymer (tapping)':        {'k_range': (1, 10),       'desc': 'Moderate stiffness, stable oscillation'},
        'Hard material (tapping)':  {'k_range': (10, 80),      'desc': 'Stiff, high resonance frequency'},
        'Force spectroscopy':       {'k_range': (0.01, 1),     'desc': 'High force sensitivity needed'},
        'High-speed imaging':       {'k_range': (1, 50),       'desc': 'High f0 for fast scanning'},
    }
    
    app = apps[application]
    T = 300
    
    print(f"  Cantilever Selection Guide")
    print(f"  ─────────────────────────────────")
    print(f"  Application: {application}")
    print(f"  {app['desc']}")
    print(f"  Recommended k range: {app['k_range'][0]:.3g} – {app['k_range'][1]:.3g} N/m")
    print()
    
    print(f"  {'Cantilever':<20s}  {'k (N/m)':>8s}  {'f0 (kHz)':>9s}  {'z_rms (nm)':>10s}  {'F_min (pN)':>10s}  {'Match':>6s}")
    print(f"  {'─'*20}  {'─'*8}  {'─'*9}  {'─'*10}  {'─'*10}  {'─'*6}")
    for name, props in cantilevers.items():
        kv = props['k']
        zrms = np.sqrt(k_B * T / kv) * 1e9
        fmin = kv * np.sqrt(k_B * T / kv) * 1e12
        match = "  ✓" if app['k_range'][0] <= kv <= app['k_range'][1] else ""
        print(f"  {name:<20s}  {kv:8.3g}  {props['f0']:9.0f}  {zrms:10.4f}  {fmin:10.2f}  {match:>6s}")
    print()

    # Plot comprehensive comparison
    k_range = np.logspace(-2, 2, 500)
    z_rms = np.sqrt(k_B * T / k_range) * 1e9
    F_min = k_range * np.sqrt(k_B * T / k_range) * 1e12  # pN
    
    # Force sensitivity (deflection per nN of force)
    sensitivity = 1 / k_range * 1e9  # nm/nN

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))

    # z_rms vs k
    axes[0,0].loglog(k_range, z_rms, 'b-', lw=2.5)
    for name, props in cantilevers.items():
        kv = props['k']
        zr = np.sqrt(k_B * T / kv) * 1e9
        c = 'green' if app['k_range'][0] <= kv <= app['k_range'][1] else 'gray'
        axes[0,0].plot(kv, zr, 'o', color=c, ms=8)
        axes[0,0].annotate(name.split('(')[0].strip(), (kv, zr), fontsize=7, rotation=15)
    axes[0,0].axvspan(app['k_range'][0], app['k_range'][1], alpha=0.15, color='green')
    axes[0,0].set_xlabel('k (N/m)')
    axes[0,0].set_ylabel('z_rms (nm)')
    axes[0,0].set_title('Thermal noise amplitude')

    # F_min vs k
    axes[0,1].loglog(k_range, F_min, 'r-', lw=2.5)
    for name, props in cantilevers.items():
        kv = props['k']
        fm = kv * np.sqrt(k_B * T / kv) * 1e12
        c = 'green' if app['k_range'][0] <= kv <= app['k_range'][1] else 'gray'
        axes[0,1].plot(kv, fm, 'o', color=c, ms=8)
    axes[0,1].axvspan(app['k_range'][0], app['k_range'][1], alpha=0.15, color='green')
    axes[0,1].set_xlabel('k (N/m)')
    axes[0,1].set_ylabel('F_min (pN)')
    axes[0,1].set_title('Minimum detectable force')

    # Sensitivity vs k
    axes[1,0].loglog(k_range, sensitivity, 'g-', lw=2.5)
    for name, props in cantilevers.items():
        kv = props['k']
        s = 1 / kv * 1e9
        c = 'green' if app['k_range'][0] <= kv <= app['k_range'][1] else 'gray'
        axes[1,0].plot(kv, s, 'o', color=c, ms=8)
    axes[1,0].axvspan(app['k_range'][0], app['k_range'][1], alpha=0.15, color='green')
    axes[1,0].set_xlabel('k (N/m)')
    axes[1,0].set_ylabel('Sensitivity (nm / nN)')
    axes[1,0].set_title('Force sensitivity (deflection per nN)')

    # SNR proxy: signal / noise ∝ 1/k / sqrt(kBT/k) ∝ 1/sqrt(k)
    # For a fixed applied force F_app = 1 nN
    F_app = 1e-9  # 1 nN
    signal = F_app / k_range * 1e9  # deflection in nm
    noise = z_rms  # nm
    snr = signal / noise
    axes[1,1].loglog(k_range, snr, 'm-', lw=2.5)
    for name, props in cantilevers.items():
        kv = props['k']
        sig = F_app / kv * 1e9
        noi = np.sqrt(k_B * T / kv) * 1e9
        c = 'green' if app['k_range'][0] <= kv <= app['k_range'][1] else 'gray'
        axes[1,1].plot(kv, sig/noi, 'o', color=c, ms=8)
    axes[1,1].axvspan(app['k_range'][0], app['k_range'][1], alpha=0.15, color='green')
    axes[1,1].set_xlabel('k (N/m)')
    axes[1,1].set_ylabel('SNR (for F = 1 nN)')
    axes[1,1].set_title('Signal-to-noise ratio')

    plt.tight_layout()
    plt.show()

interact(
    interactive_cantilever_selection,
    application=Dropdown(
        options=['Biological (contact)', 'Polymer (tapping)', 'Hard material (tapping)', 
                 'Force spectroscopy', 'High-speed imaging'],
        value='Biological (contact)',
        description='Application'
    )
);


interactive(children=(Dropdown(description='Application', options=('Biological (contact)', 'Polymer (tapping)'…

---
## Summary: Key Takeaways from Chapter 3

### 1. **Cantilever as Force Sensor** — Hooke's Law
- The cantilever converts interaction forces into measurable deflections: $F = k \, \Delta z$
- Spring constant $k$ determines force sensitivity and noise characteristics

### 2. **Optical Beam Deflection** — Geometric Amplification
- The optical lever converts tiny cantilever tilts into measurable spot displacements: $\Delta x = 2L\theta$
- Detector distance $L$ provides geometric amplification without electronic noise

### 3. **Quadrant Photodiode** — Differential Detection
- Normalized differential signals provide robust, intensity-independent detection
- Simultaneous measurement of vertical deflection and lateral (friction) forces

### 4. **Piezoelectric Scanners** — Voltage to Nanometer Motion
- Piezo actuators enable sub-angstrom positioning: $\Delta x = d \cdot V$
- Hysteresis and creep introduce positioning errors; closed-loop correction compensates

### 5. **PI Feedback Control** — Maintaining Constant Interaction
- Proportional term: fast response; Integral term: eliminates steady-state error
- Gain tuning balances tracking accuracy against oscillation and noise

### 6. **Feedback Bandwidth** — Low-Pass Filtering of Topography
- Only features with frequencies below the bandwidth are faithfully tracked
- Scan speed must be matched to feedback capability: $v_{\text{max}} = f_{\text{BW}} \times \lambda_{\text{min}}$

### 7. **Thermal Noise** — Fundamental Sensitivity Limit
- Equipartition theorem: $z_{\text{rms}} = \sqrt{k_B T / k}$
- Sets the minimum detectable force: softer cantilevers are more sensitive but noisier

### 8. **Calibration Chain** — From Voltage to Force
- Deflection sensitivity $S$ (nm/V) + spring constant $k$ (N/m) = quantitative force
- Calibration errors propagate directly into all force measurements

### 9. **Cantilever Selection** — Application-Dependent Trade-Offs
- No single cantilever is optimal for all applications
- Balance force sensitivity, thermal noise, resonance frequency, and sample requirements


---

## Notebook Complete

You have now explored **all major instrumentation topics from Chapter 3**:

1. Cantilever force–deflection (Hooke's law)
2. Optical lever amplification
3. Quadrant photodiode detection
4. Piezoelectric scanner displacement
5. Piezoelectric hysteresis and nonlinearity
6. PI feedback control
7. Feedback bandwidth and scan speed
8. Thermal noise and sensitivity limits
9. Deflection sensitivity calibration (V → Δz → F)
10. Cantilever selection and parameter trade-offs

**Next steps:**
- Try the numerical problems in Section 3.9.2 of the textbook
- Combine concepts from Chapters 2 & 3 to design a complete AFM measurement
- Explore how instrumentation choices affect the force–distance curves from Chapter 2

---

**Notebook authored for:** *Scanning Probe Microscopy* textbook, Chapter 3
